# 103 — Transformación y descomposición de consultas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** 1) ¿Quién escribió 'Cien años de soledad'? → García Márquez.
2) ¿En qué universidad estudió? → Universidad Nacional de Colombia (dep. de 1).
3) ¿Cuándo se fundó esa universidad? → 1867 (dep. de 2).
4) ¿Cuándo se publicó la novela? → 1967 (independiente; puede ir en paralelo con 2-3).
Síntesis: 1967 − 1867 = 100 años. Nótese que el paso 4 no depende de la cadena: una
buena descomposición identifica qué es secuencial y qué es paralelizable.

**Ejercicio 2.** `0.9³ = 0.729` y `0.9⁵ ≈ 0.590`. La fiabilidad cae geométricamente
con la longitud: las descomposiciones deben ser tan cortas como sea posible, y los
sistemas serios validan resultados intermedios o permiten reintentos por paso.

**Ejercicio 3.** a) **Rewriting**: hay correferencia con el historial ("el año
anterior" → "ventas de 2021"). b) **HyDE** (o multi-query): asimetría de vocabulario
paciente-literatura médica; el pasaje hipotético usa términos técnicos que acercan el
embedding al corpus. c) **Directa**: factual simple; cualquier capa extra es coste sin
ganancia. d) **Descomposición**: multi-hop (primero el ganador 2020, luego su
filmografía).

**Ejercicio 4.** El contrato se verifica en el código: `kind == "workflow"` y
`evidence` no vacía.

In [ ]:
result = run_lab("workflow", seed=103)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


In [ ]:
subpreguntas = [
    "1. ¿Quién escribió 'Cien años de soledad'?",
    "2. ¿En qué universidad estudió esa persona?      (depende de 1)",
    "3. ¿Cuándo se fundó esa universidad?             (depende de 2)",
    "4. ¿Cuándo se publicó la novela?                 (independiente)",
    "5. Síntesis: resta 4 − 3  →  1967 − 1867 = 100 años",
]
for s in subpreguntas:
    print(s)

p = 0.9
print("cadena de 3 pasos:", round(p**3, 3))
print("cadena de 5 pasos:", round(p**5, 3))

eleccion = {
    "a": "rewriting — correferencia con el historial",
    "b": "HyDE — asimetría de vocabulario pregunta/corpus técnico",
    "c": "directa — factual simple, capas extra = coste sin ganancia",
    "d": "descomposición — multi-hop: ganador 2020 → filmografía",
}
for k, v in eleccion.items():
    print(k, "→", v)

## Reflexión

1. ¿Por qué HyDE puede mejorar la recuperación incluso cuando el documento hipotético contiene datos falsos, y en qué caso concreto empeora el resultado?
2. En una descomposición de 4 pasos donde cada paso acierta con probabilidad 0.9, ¿cuál es la probabilidad aproximada de que la cadena completa sea correcta, y qué mecanismo la mejoraría?
3. ¿Qué evidencia necesitarías para justificar añadir multi-query (3 variantes) a un pipeline que ya funciona: qué métrica debería subir y qué costes deberías reportar junto a ella?